# Build the ninth investigation: 6 divides n³ − n

*Adversarial Mathematics From First Principles*, [Part III, chapter 9](https://programmer.ie/books/math/28-chapter/) · [worked example](https://programmer.ie/examples/math/28-chapter/)

Run the cells in order. The program is a small, self-contained demonstration of the chapter's mathematics; the output saved with it is what it printed when this notebook was built.

The ninth investigation is a tutorial: choose a claim and a tempting stronger
one, write the contract first, prove it, check it independently, plan how it
should fail. The claim is that 6 divides n³ − n for every natural number n;
the tempting generalization is that k divides nᵏ − n for every k ≥ 1. This is
its pinned core-Lean file, with a few reader checks added at the end.

## Run it

Colab has no Lean, so the first cell downloads the official Lean 4.35.0 release (core only, about 600 MB, a minute or two). The second writes the file and the third checks it.

In [1]:
!apt-get -qq install -y zstd > /dev/null
!test -x lean-4.35.0-linux/bin/lean || curl -sSfL https://github.com/leanprover/lean4/releases/download/v4.35.0/lean-4.35.0-linux.tar.zst | tar --zstd -x
!lean-4.35.0-linux/bin/lean --version

Lean (version 4.35.0, x86_64-unknown-linux-gnu, commit eb6cafb7290e8bd5f2937522d8d804f58f02bafc, Release)


In [2]:
%%writefile cube_tutorial.lean
/-
  Tutorial investigation T1 (Part III, chapter 9): not one of the book's
  eight investigations. Core Lean only: no Mathlib is imported. `import Lean`
  is needed by the runner's identity footer, not by the proofs.

  Claim: for every natural number n, 6 divides n³ − n.
  Tempting generalization: for every k ≥ 1 and every n, k divides nᵏ − n.
  It holds for k = 1, 2, 3, 5, 7, … but fails at k = 4, n = 2.
-/
import Lean

namespace CubeTutorial

/-- T1-P01: 6 divides n³ − n for every natural number n. -/
theorem six_dvd_cube_sub (n : Nat) : 6 ∣ n ^ 3 - n := by
  apply Nat.dvd_of_mod_eq_zero
  apply Nat.sub_mod_eq_zero_of_mod_eq
  rw [Nat.pow_mod]
  have h : n % 6 < 6 := Nat.mod_lt _ (by decide)
  generalize n % 6 = r at h ⊢
  revert r
  decide

/-- T1-P02: the witness. 2⁴ − 2 = 14, and 4 does not divide 14. -/
theorem four_not_dvd : ¬ 4 ∣ 2 ^ 4 - 2 := by decide

/-- T1-P03: the generalization is false. -/
theorem generalization_false : ¬ ∀ k n : Nat, 1 ≤ k → k ∣ n ^ k - n :=
  fun h => four_not_dvd (h 4 2 (by decide))

/-- T1-P04: the prime case k = 3, one reason the generalization is tempting. -/
theorem three_dvd_cube_sub (n : Nat) : 3 ∣ n ^ 3 - n := by
  apply Nat.dvd_of_mod_eq_zero
  apply Nat.sub_mod_eq_zero_of_mod_eq
  rw [Nat.pow_mod]
  have h : n % 3 < 3 := Nat.mod_lt _ (by decide)
  generalize n % 3 = r at h ⊢
  revert r
  decide

end CubeTutorial

-- Reader's additions (not part of the pinned file formal/lean/CubeTutorial.lean):
#print axioms CubeTutorial.six_dvd_cube_sub
#print axioms CubeTutorial.generalization_false
#eval (List.range 12).map (fun n => (n ^ 3 - n) % 6)        -- all 0
#eval (List.range 6).map (fun k => (2 ^ k - 2) % max k 1)  -- k ∣ 2^k − 2 fails first at k = 4

Writing cube_tutorial.lean


In [3]:
!lean-4.35.0-linux/bin/lean cube_tutorial.lean

'CubeTutorial.six_dvd_cube_sub' depends on axioms: [propext]
'CubeTutorial.generalization_false' depends on axioms: [propext]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 2, 0]


## What the output shows

Lean accepts the claim, the k = 3 case and the refutation of the
generalization, whose witness is k = 4, n = 2: 2⁴ − 2 = 14 is not divisible by
4. Each rests only on `propext`. The two evaluations show the claim on 0–11
and the generalization failing first at k = 4.

## Not shown

- The reader checks at the end are not part of the pinned file `formal/lean/CubeTutorial.lean`.
- The tutorial's runner record, controls and contract are in the repository (Part III, chapter 9); this file is only its proofs.

## Challenge it

Examine the tutorial's claim and its generalization. Prove 6 ∣ n³ − n by hand,
explain why k ∣ nᵏ − n holds for prime k (Fermat's little theorem) and fails
at k = 4, and design two failure controls the investigation should include.
Then choose a claim of your own and write its contract before its proof.